# Validation and cross-round checks for the revised ESS Round 4 and Round 8 datasets

This notebook validates the four principal respondent-level datasets produced by the revised curation notebooks.

The principal datasets retain all respondents aged 18 or older, as well as respondents whose age is missing. Respondents are **not** excluded because of the number of missing constructed beliefs, and missing belief values are not imputed.

The notebook performs five groups of checks:

1. validates the dimensions, schemas, respondent identifiers, adult-sample rule, missingness diagnostics, belief ranges, and official ESS weights of all four principal outputs;
2. reconstructs the historical CCA-compatible subsets using `cca_missingness_eligible == True`;
3. validates the ESS8 historical subset against Supplementary Table A2;
4. validates the ESS4 historical subset respondent by respondent against Van Noord's deterministic `df_ESS4.RData`;
5. creates descriptive comparisons of the 12 belief concepts shared by ESS4 and ESS8 using the **full principal adult samples**.

The historical CCA threshold is used only for reference validation. It is not used to define the saved principal datasets.

## Before running the notebook

The Python package `pyreadr` is required to open Van Noord's `df_ESS4.RData`.

Install it once in the same Python environment used by this Jupyter kernel:

```python
%pip install pyreadr
```

After installation, restart the kernel and run all cells.

The notebook expects the revised curation notebooks to have already created:

```text
data/processed/ess4_beliefs_all_adults_without_weights.csv
data/processed/ess4_beliefs_all_adults_with_weights.csv
data/processed/ess8_beliefs_all_adults_without_weights.csv
data/processed/ess8_beliefs_all_adults_with_weights.csv
```

It also expects:

```text
reference/van_noord/ESS Round 4/data/df_ESS4.RData
src/ess4_config.py
src/ess8_config.py
src/ess_curation_common.py
```

## Step 1 — Locate the project, import the shared configuration, and define paths

The notebook may be launched from either the project root or the `notebooks/` folder. The code below locates the project automatically and defines all required input and output paths.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if all(
            (candidate / folder).is_dir()
            for folder in ("data", "notebooks", "src")
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "data/, notebooks/, and src/."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import ess4_config as ess4
from src import ess8_config as ess8
from src.ess_curation_common import (
    summarise_beliefs,
    validate_weight_split,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

ESS4_WITHOUT_PATH = (
    PROCESSED_DIR / ess4.OUTPUT_FILENAMES["without_weights"]
)
ESS4_WITH_PATH = (
    PROCESSED_DIR / ess4.OUTPUT_FILENAMES["with_weights"]
)
ESS8_WITHOUT_PATH = (
    PROCESSED_DIR / ess8.OUTPUT_FILENAMES["without_weights"]
)
ESS8_WITH_PATH = (
    PROCESSED_DIR / ess8.OUTPUT_FILENAMES["with_weights"]
)

ESS4_REFERENCE_RDATA_PATH = PROJECT_ROOT.joinpath(
    *ess4.REFERENCE_RDATA_PARTS
)

ESS4_VALIDATION_PATH = (
    PROCESSED_DIR / ess4.OUTPUT_FILENAMES["validation"]
)
ESS8_VALIDATION_PATH = (
    PROCESSED_DIR / ess8.OUTPUT_FILENAMES["validation"]
)
CROSS_ROUND_DESCRIPTIVES_PATH = (
    PROCESSED_DIR
    / "ess4_ess8_shared_belief_descriptives.csv"
)

required_paths = (
    ESS4_WITHOUT_PATH,
    ESS4_WITH_PATH,
    ESS8_WITHOUT_PATH,
    ESS8_WITH_PATH,
    ESS4_REFERENCE_RDATA_PATH,
)

missing_paths = [
    path for path in required_paths
    if not path.exists()
]
if missing_paths:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(f"- {path}" for path in missing_paths)
    )

print("Project root:", PROJECT_ROOT)
print(
    "All four revised datasets and the "
    "ESS4 reference RData file were found."
)

Project root: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation
All four revised datasets and the ESS4 reference RData file were found.


## Step 2 — Load the four principal datasets

For each round, the weighted and unweighted versions must contain the same respondents and the same metadata, belief values, and missingness diagnostics. The only permitted difference is the presence of the four official ESS weight columns.

In [2]:
ess4_without = pd.read_csv(
    ESS4_WITHOUT_PATH,
    low_memory=False,
)
ess4_with = pd.read_csv(
    ESS4_WITH_PATH,
    low_memory=False,
)
ess8_without = pd.read_csv(
    ESS8_WITHOUT_PATH,
    low_memory=False,
)
ess8_with = pd.read_csv(
    ESS8_WITH_PATH,
    low_memory=False,
)

loaded_summary = pd.DataFrame(
    [
        {
            "dataset": "ESS4 without weights",
            "rows": len(ess4_without),
            "columns": ess4_without.shape[1],
        },
        {
            "dataset": "ESS4 with weights",
            "rows": len(ess4_with),
            "columns": ess4_with.shape[1],
        },
        {
            "dataset": "ESS8 without weights",
            "rows": len(ess8_without),
            "columns": ess8_without.shape[1],
        },
        {
            "dataset": "ESS8 with weights",
            "rows": len(ess8_with),
            "columns": ess8_with.shape[1],
        },
    ]
)

display(loaded_summary)

,dataset,rows,columns
0,ESS4 without weights,55044,35
1,ESS4 with weights,55044,39
2,ESS8 without weights,43148,37
3,ESS8 with weights,43148,41


## Step 3 — Validate the principal output schemas and sample definitions

For each round, this step checks that:

- the configured number of principal adult respondents and countries is present;
- the output columns are in the configured order;
- respondent identifiers are complete and unique;
- every retained respondent is aged 18 or older, or has missing age;
- no restriction is imposed on the number of missing beliefs;
- `n_belief_missing` and `n_belief_available` are internally consistent;
- `cca_missingness_eligible` exactly identifies respondents with no more than two missing beliefs;
- all observed belief values lie between 0 and 1;
- all four official ESS weights are present and complete;
- removing the four weight columns reproduces the unweighted file exactly.

In [3]:
def coerce_boolean_flag(
    series: pd.Series,
    *,
    column_name: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series.dtype):
        result = series.astype("boolean")
    else:
        text = (
            series.astype("string")
            .str.strip()
            .str.lower()
        )
        result = text.map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            }
        ).astype("boolean")

    if result.isna().any():
        invalid_values = sorted(
            series.loc[result.isna()]
            .astype("string")
            .dropna()
            .unique()
            .tolist()
        )
        raise AssertionError(
            f"Invalid values in {column_name}: "
            f"{invalid_values}"
        )

    return result


def validate_round_outputs(
    *,
    round_label: str,
    without_weights: pd.DataFrame,
    with_weights: pd.DataFrame,
    config,
) -> dict:
    assert (
        without_weights.shape
        == config.EXPECTED_WITHOUT_WEIGHTS_SHAPE
    )
    assert (
        with_weights.shape
        == config.EXPECTED_WITH_WEIGHTS_SHAPE
    )

    assert list(without_weights.columns) == list(
        config.OUTPUT_COLUMNS_WITHOUT_WEIGHTS
    )
    assert list(with_weights.columns) == list(
        config.OUTPUT_COLUMNS_WITH_WEIGHTS
    )

    assert len(without_weights) == config.EXPECTED_ANALYSIS_N
    assert (
        without_weights["cntry"].nunique(dropna=True)
        == config.EXPECTED_ANALYSIS_COUNTRIES
    )

    assert without_weights["ess_unique_id"].notna().all()
    assert without_weights["ess_unique_id"].is_unique
    assert without_weights["idno"].notna().all()
    assert without_weights["cntry"].notna().all()

    adult_or_missing_age = (
        without_weights["agea"].ge(config.MINIMUM_AGE)
        | without_weights["agea"].isna()
    )
    assert adult_or_missing_age.all()

    expected_missing = (
        without_weights.loc[:, config.BELIEF_COLUMNS]
        .isna()
        .sum(axis=1)
    )
    assert without_weights["n_belief_missing"].eq(
        expected_missing
    ).all()
    assert without_weights["n_belief_available"].eq(
        len(config.BELIEF_COLUMNS) - expected_missing
    ).all()

    cca_flag = coerce_boolean_flag(
        without_weights["cca_missingness_eligible"],
        column_name="cca_missingness_eligible",
    )
    expected_cca_flag = expected_missing.le(
        config.CCA_MAXIMUM_MISSING_BELIEFS
    )
    assert cca_flag.eq(expected_cca_flag).all()
    assert int(cca_flag.sum()) == config.EXPECTED_CCA_N

    belief_values = without_weights.loc[
        :, config.BELIEF_COLUMNS
    ]
    observed_minimum = (
        belief_values.min(skipna=True).min()
    )
    observed_maximum = (
        belief_values.max(skipna=True).max()
    )
    assert observed_minimum >= 0.0
    assert observed_maximum <= 1.0

    validate_weight_split(
        without_weights,
        with_weights,
        config.WEIGHT_COLUMNS,
        require_complete_weights=True,
    )

    return {
        "round": round_label,
        "principal_respondents": len(without_weights),
        "countries": without_weights["cntry"].nunique(
            dropna=True
        ),
        "beliefs": len(config.BELIEF_COLUMNS),
        "historical_cca_eligible": int(cca_flag.sum()),
        "retained_above_cca_missingness_threshold": int(
            (~cca_flag).sum()
        ),
        "minimum_observed_belief": observed_minimum,
        "maximum_observed_belief": observed_maximum,
        "maximum_missing_beliefs_observed": int(
            expected_missing.max()
        ),
        "weights_complete": bool(
            with_weights.loc[:, config.WEIGHT_COLUMNS]
            .notna()
            .all()
            .all()
        ),
        "weighted_unweighted_match": True,
    }


round_checks = pd.DataFrame(
    [
        validate_round_outputs(
            round_label=ess4.ROUND_LABEL,
            without_weights=ess4_without,
            with_weights=ess4_with,
            config=ess4,
        ),
        validate_round_outputs(
            round_label=ess8.ROUND_LABEL,
            without_weights=ess8_without,
            with_weights=ess8_with,
            config=ess8,
        ),
    ]
)

display(round_checks)

,round,principal_respondents,countries,beliefs,historical_cca_eligible,retained_above_cca_missingness_threshold,minimum_observed_belief,maximum_observed_belief,maximum_missing_beliefs_observed,weights_complete,weighted_unweighted_match
0,ESS Round 4,55044,29,19,45268,9776,0.0,1.0,19,True,True
1,ESS Round 8,43148,23,20,37118,6030,0.0,1.0,20,True,True


## Step 4 — Inspect belief-missingness distributions in the full adult samples

These tables show how many respondents have each possible number of missing constructed beliefs. They confirm transparently that respondents with more than two missing beliefs remain in the principal datasets.

In [4]:
def missingness_distribution(
    dataframe: pd.DataFrame,
    *,
    round_label: str,
) -> pd.DataFrame:
    result = (
        dataframe["n_belief_missing"]
        .value_counts()
        .sort_index()
        .rename_axis("n_belief_missing")
        .rename("respondents")
        .reset_index()
    )
    result.insert(0, "round", round_label)
    result["percent"] = (
        100.0
        * result["respondents"]
        / len(dataframe)
    )
    result["historical_cca_eligible"] = (
        result["n_belief_missing"] <= 2
    )
    return result


missingness_tables = pd.concat(
    [
        missingness_distribution(
            ess4_without,
            round_label=ess4.ROUND_LABEL,
        ),
        missingness_distribution(
            ess8_without,
            round_label=ess8.ROUND_LABEL,
        ),
    ],
    ignore_index=True,
)

display(missingness_tables)

,round,n_belief_missing,respondents,percent,historical_cca_eligible
0,ESS Round 4,0,30126,54.730761,True
1,ESS Round 4,1,9561,17.369741,True
2,ESS Round 4,2,5581,10.139161,True
3,ESS Round 4,3,3068,5.573723,False
4,ESS Round 4,4,1993,3.620740,False
5,ESS Round 4,5,1373,2.494368,False
6,ESS Round 4,6,908,1.649589,False
7,ESS Round 4,7,662,1.202674,False
8,ESS Round 4,8,514,0.933798,False
9,ESS Round 4,9,346,0.628588,False


## Step 5 — Reconstruct the historical CCA-compatible subsets

The historical subsets are obtained from the saved principal datasets using the eligibility flag. These subsets are used only for comparison with the historical ESS8 publication targets and the ESS4 R reference.

The expected historical sample sizes are:

- ESS4: 45,268 respondents;
- ESS8: 37,118 respondents.

In [5]:
ess4_cca_flag = coerce_boolean_flag(
    ess4_without["cca_missingness_eligible"],
    column_name="ESS4 cca_missingness_eligible",
)
ess8_cca_flag = coerce_boolean_flag(
    ess8_without["cca_missingness_eligible"],
    column_name="ESS8 cca_missingness_eligible",
)

ess4_cca_subset = (
    ess4_without.loc[ess4_cca_flag]
    .copy()
    .reset_index(drop=True)
)
ess8_cca_subset = (
    ess8_without.loc[ess8_cca_flag]
    .copy()
    .reset_index(drop=True)
)

assert len(ess4_cca_subset) == ess4.EXPECTED_CCA_N
assert len(ess8_cca_subset) == ess8.EXPECTED_CCA_N
assert (
    ess4_cca_subset["cntry"].nunique(dropna=True)
    == ess4.EXPECTED_CCA_COUNTRIES
)
assert (
    ess8_cca_subset["cntry"].nunique(dropna=True)
    == ess8.EXPECTED_CCA_COUNTRIES
)
assert ess4_cca_subset["n_belief_missing"].le(
    ess4.CCA_MAXIMUM_MISSING_BELIEFS
).all()
assert ess8_cca_subset["n_belief_missing"].le(
    ess8.CCA_MAXIMUM_MISSING_BELIEFS
).all()

subset_summary = pd.DataFrame(
    [
        {
            "round": ess4.ROUND_LABEL,
            "principal_adult_sample": len(ess4_without),
            "historical_cca_subset": len(ess4_cca_subset),
            "retained_outside_cca_subset": (
                len(ess4_without) - len(ess4_cca_subset)
            ),
        },
        {
            "round": ess8.ROUND_LABEL,
            "principal_adult_sample": len(ess8_without),
            "historical_cca_subset": len(ess8_cca_subset),
            "retained_outside_cca_subset": (
                len(ess8_without) - len(ess8_cca_subset)
            ),
        },
    ]
)

display(subset_summary)

,round,principal_adult_sample,historical_cca_subset,retained_outside_cca_subset
0,ESS Round 4,55044,45268,9776
1,ESS Round 8,43148,37118,6030


## Step 6 — Validate the ESS8 historical subset against Supplementary Table A2

Supplementary Table A2 corresponds to the historical CCA-compatible ESS8 sample. Therefore, the published non-missing counts, means, and standard deviations are compared with `ess8_cca_subset`, not with the full 43,148-person principal dataset.

The published means and standard deviations are reported to two decimal places, so they are compared after rounding to two decimal places. Non-missing counts must match exactly.

In [6]:
ess8_reproduced = summarise_beliefs(
    ess8_cca_subset,
    ess8.BELIEF_COLUMNS,
)

ess8_paper = pd.DataFrame(
    ess8.PAPER_STATISTICS,
    columns=ess8.PAPER_STATISTICS_COLUMNS,
)

ess8_validation = ess8_paper.merge(
    ess8_reproduced,
    on="belief_variable",
    how="left",
    validate="one_to_one",
)

ess8_validation["mean_reproduced_round2"] = (
    ess8_validation["mean_reproduced"].round(2)
)
ess8_validation["sd_reproduced_round2"] = (
    ess8_validation["sd_reproduced"].round(2)
)
ess8_validation["N_matches"] = (
    ess8_validation["N_paper"]
    == ess8_validation["N_reproduced"]
)
ess8_validation["mean_matches_round2"] = (
    ess8_validation["mean_paper"]
    == ess8_validation["mean_reproduced_round2"]
)
ess8_validation["sd_matches_round2"] = (
    ess8_validation["sd_paper"]
    == ess8_validation["sd_reproduced_round2"]
)

match_columns = [
    "N_matches",
    "mean_matches_round2",
    "sd_matches_round2",
]
assert ess8_validation.loc[
    :, match_columns
].all().all()

ess8_validation.to_csv(
    ESS8_VALIDATION_PATH,
    index=False,
)

print(
    "All 20 ESS8 belief variables in the "
    "historical CCA subset match "
    "Supplementary Table A2."
)
print(
    "Validation file written to:",
    ESS8_VALIDATION_PATH,
)
display(ess8_validation)

All 20 ESS8 belief variables in the historical CCA subset match Supplementary Table A2.
Validation file written to: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess8_cca_subset_validation_against_paper.csv


,belief_variable,N_paper,mean_paper,sd_paper,N_reproduced,mean_reproduced,sd_reproduced,mean_reproduced_round2,sd_reproduced_round2,N_matches,mean_matches_round2,sd_matches_round2
0,left_right_identification,34248,0.51,0.22,34248,0.512920,0.223370,0.51,0.22,True,True,True
1,gender_inequality,37038,0.23,0.27,37038,0.227955,0.269111,0.23,0.27,True,True,True
2,anti_lgbt,36098,0.34,0.27,36098,0.336637,0.270381,0.34,0.27,True,True,True
3,euroscepticism,35848,0.51,0.27,35848,0.508968,0.266828,0.51,0.27,True,True,True
4,anti_immigration,36459,0.45,0.27,36459,0.451734,0.265234,0.45,0.27,True,True,True
5,anti_egalitarianism,36772,0.38,0.19,36772,0.380239,0.193550,0.38,0.19,True,True,True
6,benefits_harm_economy,35956,0.49,0.23,35956,0.490534,0.226702,0.49,0.23,True,True,True
7,benefits_harm_society,36801,0.41,0.22,36801,0.410132,0.218079,0.41,0.22,True,True,True
8,welfare_chauvinism,36441,0.54,0.26,36441,0.543090,0.258760,0.54,0.26,True,True,True
9,anti_economic_interventionism,36927,0.24,0.16,36927,0.244343,0.159874,0.24,0.16,True,True,True


## Step 7 — Load Van Noord's deterministic ESS4 reference object

`df_ESS4.RData` is the saved output of the supplied `Data cleaning_ESS4.R` script. It contains the historical 45,268-person CCA-compatible ESS4 sample.

The code accepts either an R object named `df` or a file containing a single data-frame object.

In [7]:
try:
    import pyreadr
except ImportError as exc:
    raise ImportError(
        "pyreadr is required for this notebook. "
        "Run `%pip install pyreadr`, restart the "
        "kernel, and run the notebook again."
    ) from exc

r_objects = pyreadr.read_r(
    str(ESS4_REFERENCE_RDATA_PATH)
)

if "df" in r_objects:
    ess4_reference_raw = r_objects["df"]
elif len(r_objects) == 1:
    ess4_reference_raw = next(
        iter(r_objects.values())
    )
else:
    raise KeyError(
        "Could not identify the ESS4 reference "
        "dataframe inside the RData file. "
        f"Objects found: {list(r_objects)}"
    )

assert isinstance(
    ess4_reference_raw,
    pd.DataFrame,
)
assert (
    len(ess4_reference_raw)
    == ess4.EXPECTED_CCA_N
)

print("Objects found in RData:", list(r_objects))
print(
    "Reference rows:",
    len(ess4_reference_raw),
)
print(
    "Reference columns:",
    ess4_reference_raw.shape[1],
)
display(ess4_reference_raw.head())

Objects found in RData: ['df']
Reference rows: 45268
Reference columns: 29


,id,essid,country,education,hhincome,female,age,religious,urbanization,ethnic_minority,...,anti_interventionism,harsh_sentences,anti_mil_democracy,science_environment,government_spending,regressive_taxes,regressive_benefits,age_prejudice,authoritarianism,anti_libertarianism
0,1,10202.0,BE,Higher educated,4.0,Male,36.0,Non-religious,5.0,Ethnic majority,...,0.316667,0.25,0.75,0.25,0.5,0.5,0.0,0.6,0.48,0.40
1,2,10203.0,BE,Higher educated,7.0,Female,26.0,Non-religious,5.0,Ethnic majority,...,0.300000,0.75,0.25,0.50,0.5,0.5,0.0,0.3,0.60,0.44
2,3,10207.0,BE,Higher educated,10.0,Male,69.0,Religious,5.0,Ethnic majority,...,0.550000,0.50,0.75,0.75,0.5,0.0,0.0,0.6,0.56,0.20
3,4,10208.0,BE,Higher educated,7.0,Female,77.0,Religious,5.0,Ethnic majority,...,0.500000,0.50,0.25,0.25,0.5,0.0,0.0,0.2,0.68,0.44
4,5,10302.0,BE,Middle educated,7.0,Male,27.0,Non-religious,5.0,Ethnic majority,...,0.450000,0.25,0.75,0.25,0.5,0.5,0.5,0.0,0.64,0.36


## Step 8 — Harmonize the ESS4 reference names and factor labels

Van Noord's R object uses different column names and stores several demographics as labelled factors.

This step maps the R object to the Python conventions without changing the underlying responses:

- `essid` → `idno`;
- `country` → `cntry`;
- factor labels for education, gender, religion, and ethnic-minority status are mapped to the numeric categories used in the Python output;
- all 19 R belief columns are renamed to the corresponding Python names.

In [8]:
R_TO_PYTHON_BELIEF = {
    "lrscale": "left_right_identification",
    "gender_inequality": "gender_inequality",
    "anti_lgbt": "anti_lgbt",
    "euroscepticism": "euroscepticism",
    "anti_immigration": "anti_immigration",
    "anti_egalitarianism": "anti_egalitarianism",
    "benefits_eco": "benefits_harm_economy",
    "benefits_soc": "benefits_harm_society",
    "welfare_chauvinism": "welfare_chauvinism",
    "anti_interventionism": (
        "anti_economic_interventionism"
    ),
    "harsh_sentences": "harsh_sentences",
    "anti_mil_democracy": "anti_militant_democracy",
    "science_environment": (
        "no_science_environment_solution"
    ),
    "government_spending": "anti_government_spending",
    "regressive_taxes": "regressive_taxes",
    "regressive_benefits": "regressive_benefits",
    "age_prejudice": "age_prejudice",
    "authoritarianism": "authoritarianism",
    "anti_libertarianism": "anti_libertarianism",
}

required_reference_columns = {
    "essid",
    "country",
    "education",
    "hhincome",
    "female",
    "age",
    "religious",
    "urbanization",
    "ethnic_minority",
    *R_TO_PYTHON_BELIEF.keys(),
}

missing_reference_columns = sorted(
    required_reference_columns
    - set(ess4_reference_raw.columns)
)
if missing_reference_columns:
    raise KeyError(
        "Required columns are missing from "
        "df_ESS4.RData: "
        f"{missing_reference_columns}"
    )


def map_factor_to_numeric(
    series: pd.Series,
    *,
    label_mapping: dict[str, float],
    numeric_code_mapping: dict[float, float],
) -> pd.Series:
    text_values = (
        series.astype("string").str.strip()
    )
    from_labels = text_values.map(label_mapping)

    numeric_values = pd.to_numeric(
        series,
        errors="coerce",
    )
    from_codes = numeric_values.map(
        numeric_code_mapping
    )

    return pd.to_numeric(
        from_labels.fillna(from_codes),
        errors="coerce",
    )


country_text = (
    ess4_reference_raw["country"]
    .astype("string")
    .str.strip()
)
country_name_to_code = {
    name: code
    for code, name in ess4.COUNTRY_LABELS.items()
}
country_name_to_code.update(
    {
        "Great Britain": "GB",
        "United Kingdom": "GB",
        "Czech Republic": "CZ",
        "Czechia": "CZ",
        "Russia": "RU",
        "Russian Federation": "RU",
        "Turkey": "TR",
        "Türkiye": "TR",
    }
)

ess4_reference = pd.DataFrame(
    index=ess4_reference_raw.index
)
ess4_reference["idno"] = pd.to_numeric(
    ess4_reference_raw["essid"],
    errors="coerce",
).astype("Int64")
ess4_reference["cntry"] = country_text.where(
    country_text.isin(ess4.COUNTRY_LABELS),
    country_text.map(country_name_to_code),
)

ess4_reference["education_3cat"] = (
    map_factor_to_numeric(
        ess4_reference_raw["education"],
        label_mapping={
            "Lower educated": 1,
            "Middle educated": 2,
            "Higher educated": 3,
        },
        numeric_code_mapping={
            1: 1,
            2: 2,
            3: 3,
        },
    )
)
ess4_reference["hinctnta"] = pd.to_numeric(
    ess4_reference_raw["hhincome"],
    errors="coerce",
)
ess4_reference["gndr"] = map_factor_to_numeric(
    ess4_reference_raw["female"],
    label_mapping={
        "Male": 1,
        "Female": 2,
    },
    numeric_code_mapping={
        1: 1,
        2: 2,
    },
)
ess4_reference["agea"] = pd.to_numeric(
    ess4_reference_raw["age"],
    errors="coerce",
)
ess4_reference["rlgblg"] = map_factor_to_numeric(
    ess4_reference_raw["religious"],
    label_mapping={
        "Non-religious": 2,
        "Religious": 1,
    },
    numeric_code_mapping={
        1: 2,
        2: 1,
    },
)
ess4_reference["urbanization"] = pd.to_numeric(
    ess4_reference_raw["urbanization"],
    errors="coerce",
)
ess4_reference["blgetmg"] = map_factor_to_numeric(
    ess4_reference_raw["ethnic_minority"],
    label_mapping={
        "Ethnic majority": 2,
        "Ethnic minority": 1,
    },
    numeric_code_mapping={
        1: 2,
        2: 1,
    },
)

for r_column, python_column in (
    R_TO_PYTHON_BELIEF.items()
):
    ess4_reference[python_column] = pd.to_numeric(
        ess4_reference_raw[r_column],
        errors="coerce",
    )

assert ess4_reference["idno"].notna().all()
assert ess4_reference["cntry"].notna().all()
assert (
    ess4_reference[["cntry", "idno"]]
    .duplicated()
    .sum()
    == 0
)

print(
    "Harmonized ESS4 reference dimensions:",
    ess4_reference.shape,
)
display(ess4_reference.head())

Harmonized ESS4 reference dimensions: (45268, 28)


,idno,cntry,education_3cat,hinctnta,gndr,agea,rlgblg,urbanization,blgetmg,left_right_identification,...,anti_economic_interventionism,harsh_sentences,anti_militant_democracy,no_science_environment_solution,anti_government_spending,regressive_taxes,regressive_benefits,age_prejudice,authoritarianism,anti_libertarianism
0,10202,BE,3.0,4.0,1.0,36.0,2.0,5.0,2.0,0.7,...,0.316667,0.25,0.75,0.25,0.5,0.5,0.0,0.6,0.48,0.40
1,10203,BE,3.0,7.0,2.0,26.0,2.0,5.0,2.0,0.6,...,0.300000,0.75,0.25,0.50,0.5,0.5,0.0,0.3,0.60,0.44
2,10207,BE,3.0,10.0,1.0,69.0,1.0,5.0,2.0,0.8,...,0.550000,0.50,0.75,0.75,0.5,0.0,0.0,0.6,0.56,0.20
3,10208,BE,3.0,7.0,2.0,77.0,1.0,5.0,2.0,0.6,...,0.500000,0.50,0.25,0.25,0.5,0.0,0.0,0.2,0.68,0.44
4,10302,BE,2.0,7.0,1.0,27.0,2.0,5.0,2.0,0.5,...,0.450000,0.25,0.75,0.25,0.5,0.5,0.5,0.0,0.64,0.36


## Step 9 — Confirm that the historical Python and R subsets contain the same ESS4 respondents

Only the 45,268 Python respondents with `cca_missingness_eligible == True` are compared with `df_ESS4.RData`.

The comparison uses `(cntry, idno)` as the stable respondent key. Row order alone is not treated as evidence of agreement.

In [9]:
KEY_COLUMNS = ["cntry", "idno"]

ess4_python_for_comparison = (
    ess4_cca_subset.copy()
)
ess4_python_for_comparison["idno"] = (
    pd.to_numeric(
        ess4_python_for_comparison["idno"],
        errors="coerce",
    ).astype("Int64")
)
ess4_python_for_comparison["cntry"] = (
    ess4_python_for_comparison["cntry"]
    .astype("string")
    .str.strip()
)

assert (
    ess4_python_for_comparison[KEY_COLUMNS]
    .duplicated()
    .sum()
    == 0
)

respondent_key_check = (
    ess4_python_for_comparison[KEY_COLUMNS]
    .merge(
        ess4_reference[KEY_COLUMNS],
        on=KEY_COLUMNS,
        how="outer",
        indicator=True,
        validate="one_to_one",
    )
)

respondent_key_counts = (
    respondent_key_check["_merge"]
    .value_counts()
    .rename_axis("merge_status")
    .rename("respondents")
    .to_frame()
)

display(respondent_key_counts)

python_only = respondent_key_check.loc[
    respondent_key_check["_merge"] == "left_only",
    KEY_COLUMNS,
]
reference_only = respondent_key_check.loc[
    respondent_key_check["_merge"] == "right_only",
    KEY_COLUMNS,
]

if (
    not python_only.empty
    or not reference_only.empty
):
    print("First Python-only respondent keys:")
    display(python_only.head(20))
    print(
        "First R-reference-only respondent keys:"
    )
    display(reference_only.head(20))
    raise AssertionError(
        "The historical Python and R ESS4 "
        "subsets do not contain the same "
        "respondent set."
    )

print(
    "Respondent-set validation passed: "
    f"all {len(ess4_python_for_comparison):,} "
    "historical ESS4 respondents match."
)

,respondents
merge_status,
both,45268
left_only,0
right_only,0


Respondent-set validation passed: all 45,268 historical ESS4 respondents match.


## Step 10 — Compare the harmonized ESS4 demographics

This diagnostic check confirms that the demographic fields in the historical Python subset agree with the corresponding fields in Van Noord's R object after factor-label harmonization.

In [10]:
comparison = (
    ess4_python_for_comparison.merge(
        ess4_reference,
        on=KEY_COLUMNS,
        how="inner",
        suffixes=("_python", "_reference"),
        validate="one_to_one",
    )
)

DEMOGRAPHIC_COLUMNS_TO_COMPARE = (
    "education_3cat",
    "hinctnta",
    "gndr",
    "agea",
    "rlgblg",
    "urbanization",
    "blgetmg",
)


def numeric_match_mask(
    python_values: pd.Series,
    reference_values: pd.Series,
    *,
    atol: float = 1e-12,
) -> pd.Series:
    python_numeric = pd.to_numeric(
        python_values,
        errors="coerce",
    )
    reference_numeric = pd.to_numeric(
        reference_values,
        errors="coerce",
    )

    both_missing = (
        python_numeric.isna()
        & reference_numeric.isna()
    )
    both_present = (
        python_numeric.notna()
        & reference_numeric.notna()
    )
    close = pd.Series(
        np.isclose(
            python_numeric.fillna(0.0),
            reference_numeric.fillna(0.0),
            rtol=0.0,
            atol=atol,
        ),
        index=python_numeric.index,
    )

    return both_missing | (
        both_present & close
    )


demographic_validation_rows = []
for column in DEMOGRAPHIC_COLUMNS_TO_COMPARE:
    match_mask = numeric_match_mask(
        comparison[f"{column}_python"],
        comparison[f"{column}_reference"],
    )
    demographic_validation_rows.append(
        {
            "variable": column,
            "respondents_compared": len(comparison),
            "mismatches": int(
                (~match_mask).sum()
            ),
            "all_values_match": bool(
                match_mask.all()
            ),
        }
    )

demographic_validation = pd.DataFrame(
    demographic_validation_rows
)

display(demographic_validation)

assert demographic_validation[
    "all_values_match"
].all()

print(
    "All harmonized ESS4 demographic "
    "variables in the historical subset "
    "match the R reference."
)

,variable,respondents_compared,mismatches,all_values_match
0,education_3cat,45268,0,True
1,hinctnta,45268,0,True
2,gndr,45268,0,True
3,agea,45268,0,True
4,rlgblg,45268,0,True
5,urbanization,45268,0,True
6,blgetmg,45268,0,True


All harmonized ESS4 demographic variables in the historical subset match the R reference.


## Step 11 — Compare all 19 ESS4 beliefs with the R reference

For every belief in the historical subset, this step compares:

- non-missing count;
- mean;
- sample standard deviation;
- respondent-level missingness pattern;
- respondent-level numeric value, using an absolute tolerance of `1e-12`.

The resulting validation table is saved as:

```text
data/processed/ess4_cca_subset_validation_against_reference.csv
```

In [11]:
ESS4_NUMERIC_TOLERANCE = 1e-12

ess4_validation_rows = []

for belief in ess4.BELIEF_COLUMNS:
    python_values = pd.to_numeric(
        comparison[f"{belief}_python"],
        errors="coerce",
    )
    reference_values = pd.to_numeric(
        comparison[f"{belief}_reference"],
        errors="coerce",
    )

    match_mask = numeric_match_mask(
        python_values,
        reference_values,
        atol=ESS4_NUMERIC_TOLERANCE,
    )

    both_present = (
        python_values.notna()
        & reference_values.notna()
    )
    absolute_differences = (
        python_values.loc[both_present]
        - reference_values.loc[both_present]
    ).abs()

    maximum_absolute_difference = (
        float(absolute_differences.max())
        if not absolute_differences.empty
        else 0.0
    )

    ess4_validation_rows.append(
        {
            "belief_variable": belief,
            "N_python": int(
                python_values.notna().sum()
            ),
            "N_reference": int(
                reference_values.notna().sum()
            ),
            "mean_python": python_values.mean(),
            "mean_reference": (
                reference_values.mean()
            ),
            "sd_python": python_values.std(ddof=1),
            "sd_reference": (
                reference_values.std(ddof=1)
            ),
            "missingness_mismatches": int(
                (
                    python_values.isna()
                    != reference_values.isna()
                ).sum()
            ),
            "value_mismatches": int(
                (~match_mask).sum()
            ),
            "max_abs_difference": (
                maximum_absolute_difference
            ),
            "N_matches": bool(
                python_values.notna().sum()
                == reference_values.notna().sum()
            ),
            "values_match_within_tolerance": bool(
                match_mask.all()
            ),
        }
    )

ess4_validation = pd.DataFrame(
    ess4_validation_rows
)

assert ess4_validation["N_matches"].all()
assert ess4_validation[
    "values_match_within_tolerance"
].all()
assert (
    ess4_validation["missingness_mismatches"]
    == 0
).all()
assert (
    ess4_validation["max_abs_difference"]
    <= ESS4_NUMERIC_TOLERANCE
).all()

ess4_validation.to_csv(
    ESS4_VALIDATION_PATH,
    index=False,
)

print(
    "All 19 ESS4 belief variables in the "
    "historical CCA subset match "
    "df_ESS4.RData respondent by respondent."
)
print(
    "Validation file written to:",
    ESS4_VALIDATION_PATH,
)
display(ess4_validation)

All 19 ESS4 belief variables in the historical CCA subset match df_ESS4.RData respondent by respondent.
Validation file written to: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess4_cca_subset_validation_against_reference.csv


,belief_variable,N_python,N_reference,mean_python,mean_reference,sd_python,sd_reference,missingness_mismatches,value_mismatches,max_abs_difference,N_matches,values_match_within_tolerance
0,left_right_identification,41465,41465,0.518816,0.518816,0.225549,0.225549,0,0,0.000000e+00,True,True
1,gender_inequality,45010,45010,0.566380,0.566380,0.257668,0.257668,0,0,0.000000e+00,True,True
2,anti_lgbt,44516,44516,0.335346,0.335346,0.310386,0.310386,0,0,0.000000e+00,True,True
3,euroscepticism,43342,43342,0.466132,0.466132,0.263316,0.263316,0,0,0.000000e+00,True,True
4,anti_immigration,44297,44297,0.481713,0.481713,0.269713,0.269713,0,0,1.110223e-16,True,True
5,anti_egalitarianism,44984,44984,0.307890,0.307890,0.212086,0.212086,0,0,0.000000e+00,True,True
6,benefits_harm_economy,43533,43533,0.515526,0.515526,0.225427,0.225427,0,0,0.000000e+00,True,True
7,benefits_harm_society,44693,44693,0.419406,0.419406,0.223687,0.223687,0,0,0.000000e+00,True,True
8,welfare_chauvinism,44309,44309,0.563802,0.563802,0.252744,0.252744,0,0,0.000000e+00,True,True
9,anti_economic_interventionism,44656,44656,0.229808,0.229808,0.158325,0.158325,0,0,2.220446e-16,True,True


## Step 12 — Compare the 12 belief concepts shared by ESS4 and ESS8

The cross-round descriptives use the **full principal adult samples**:

- ESS4: 55,044 respondents;
- ESS8: 43,148 respondents.

The item modules are not identical across rounds, so the comparison is restricted to the intersection of the two belief lists. Differences in means, standard deviations, and missingness are descriptive and are not treated as validation failures.

In [12]:
SHARED_BELIEFS = tuple(
    belief
    for belief in ess8.BELIEF_COLUMNS
    if belief in ess4.BELIEF_COLUMNS
)

assert len(SHARED_BELIEFS) == 12

print(
    "Shared belief concepts:",
    len(SHARED_BELIEFS),
)
for belief in SHARED_BELIEFS:
    print("-", belief)


def shared_belief_descriptives(
    dataframe: pd.DataFrame,
    beliefs: tuple[str, ...],
    *,
    round_label: str,
) -> pd.DataFrame:
    result = summarise_beliefs(
        dataframe,
        beliefs,
    ).rename(
        columns={
            "N_reproduced": "N_nonmissing",
            "mean_reproduced": "mean",
            "sd_reproduced": "sd",
        }
    )
    result.insert(0, "round", round_label)
    result["N_total"] = len(dataframe)
    result["N_missing"] = (
        result["N_total"]
        - result["N_nonmissing"]
    )
    result["percent_missing"] = (
        100.0
        * result["N_missing"]
        / result["N_total"]
    )
    return result


cross_round_descriptives = pd.concat(
    [
        shared_belief_descriptives(
            ess4_without,
            SHARED_BELIEFS,
            round_label=ess4.ROUND_LABEL,
        ),
        shared_belief_descriptives(
            ess8_without,
            SHARED_BELIEFS,
            round_label=ess8.ROUND_LABEL,
        ),
    ],
    ignore_index=True,
)

cross_round_descriptives.to_csv(
    CROSS_ROUND_DESCRIPTIVES_PATH,
    index=False,
)

print(
    "Cross-round descriptive table "
    "written to:",
    CROSS_ROUND_DESCRIPTIVES_PATH,
)
display(cross_round_descriptives)

Shared belief concepts: 12
- left_right_identification
- gender_inequality
- anti_lgbt
- euroscepticism
- anti_immigration
- anti_egalitarianism
- benefits_harm_economy
- benefits_harm_society
- welfare_chauvinism
- anti_economic_interventionism
- authoritarianism
- anti_libertarianism
Cross-round descriptive table written to: /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess4_ess8_shared_belief_descriptives.csv


,round,belief_variable,N_nonmissing,mean,sd,N_total,N_missing,percent_missing
0,ESS Round 4,left_right_identification,46413,0.519755,0.228265,55044,8631,15.680183
1,ESS Round 4,gender_inequality,53539,0.547302,0.260615,55044,1505,2.734176
2,ESS Round 4,anti_lgbt,51621,0.356798,0.320101,55044,3423,6.218661
3,ESS Round 4,euroscepticism,48543,0.464296,0.266711,55044,6501,11.810552
4,ESS Round 4,anti_immigration,51377,0.489335,0.276050,55044,3667,6.661943
5,ESS Round 4,anti_egalitarianism,53152,0.298446,0.210017,55044,1892,3.437250
6,ESS Round 4,benefits_harm_economy,48729,0.511923,0.226657,55044,6315,11.472640
7,ESS Round 4,benefits_harm_society,51915,0.427153,0.228826,55044,3129,5.684543
8,ESS Round 4,welfare_chauvinism,50826,0.568739,0.255371,55044,4218,7.662961
9,ESS Round 4,anti_economic_interventionism,52717,0.219460,0.161449,55044,2327,4.227527


## Step 13 — Inspect country coverage in the full adult samples

Round 4 and Round 8 contain different country sets. This table reports the principal adult sample size in each round and should be considered when interpreting cross-round descriptive differences.

In [13]:
ess4_country_counts = (
    ess4_without
    .groupby(
        ["cntry", "country_name"],
        dropna=False,
    )
    .size()
    .rename("N_ESS4")
    .reset_index()
)

ess8_country_counts = (
    ess8_without
    .groupby(
        ["cntry", "country_name"],
        dropna=False,
    )
    .size()
    .rename("N_ESS8")
    .reset_index()
)

country_coverage = (
    ess4_country_counts.merge(
        ess8_country_counts,
        on=["cntry", "country_name"],
        how="outer",
        validate="one_to_one",
    )
    .sort_values(
        "cntry",
        kind="stable",
    )
)

country_coverage["present_in_ESS4"] = (
    country_coverage["N_ESS4"].notna()
)
country_coverage["present_in_ESS8"] = (
    country_coverage["N_ESS8"].notna()
)

display(
    country_coverage.reset_index(drop=True)
)

,cntry,country_name,N_ESS4,N_ESS8,present_in_ESS4,present_in_ESS8
0,AT,Austria,NaN,1994.0,False,True
1,BE,Belgium,1682.0,1702.0,True,True
2,BG,Bulgaria,2190.0,NaN,True,False
3,CH,Switzerland,1776.0,1465.0,True,True
4,CY,Cyprus,1168.0,NaN,True,False
5,CZ,Czechia,1955.0,2186.0,True,True
6,DE,Germany,2691.0,2726.0,True,True
7,DK,Denmark,1550.0,NaN,True,False
8,EE,Estonia,1610.0,1963.0,True,True
9,ES,Spain,2486.0,1918.0,True,True


## Step 14 — Final validation summary

A successful run means that:

- all four principal files have the revised dimensions and schemas;
- the adult-or-missing-age rule is satisfied;
- respondents are not filtered by belief missingness;
- missingness counts and the historical CCA-eligibility flag are correct;
- the official ESS weights are complete and do not alter the respondent-level data;
- the ESS8 historical subset reproduces Supplementary Table A2;
- the ESS4 historical subset agrees respondent by respondent with `df_ESS4.RData`;
- the full-sample shared-belief descriptive table has been written successfully.

In [14]:
final_summary = pd.DataFrame(
    [
        {
            "check": (
                "ESS4 principal weighted/unweighted "
                "internal validation"
            ),
            "status": "passed",
        },
        {
            "check": (
                "ESS8 principal weighted/unweighted "
                "internal validation"
            ),
            "status": "passed",
        },
        {
            "check": (
                "ESS4/ESS8 revised missingness "
                "handling and CCA flags"
            ),
            "status": "passed",
        },
        {
            "check": (
                "ESS8 historical CCA subset against "
                "Supplementary Table A2"
            ),
            "status": "passed",
        },
        {
            "check": (
                "ESS4 historical respondent subset "
                "against df_ESS4.RData"
            ),
            "status": "passed",
        },
        {
            "check": (
                "ESS4 historical demographics "
                "against df_ESS4.RData"
            ),
            "status": "passed",
        },
        {
            "check": (
                "ESS4 historical beliefs against "
                "df_ESS4.RData"
            ),
            "status": "passed",
        },
        {
            "check": (
                "ESS4/ESS8 full-sample shared-belief "
                "descriptive export"
            ),
            "status": "passed",
        },
    ]
)

display(final_summary)

print("Validation completed successfully.")
print()
print("Principal sample sizes:")
print(
    f"- ESS4: {len(ess4_without):,} respondents"
)
print(
    f"- ESS8: {len(ess8_without):,} respondents"
)
print()
print("Historical validation subset sizes:")
print(
    f"- ESS4: {len(ess4_cca_subset):,} respondents"
)
print(
    f"- ESS8: {len(ess8_cca_subset):,} respondents"
)
print()
print("Generated validation files:")
print("-", ESS4_VALIDATION_PATH)
print("-", ESS8_VALIDATION_PATH)
print("-", CROSS_ROUND_DESCRIPTIVES_PATH)

,check,status
0,ESS4 principal weighted/unweighted internal va...,passed
1,ESS8 principal weighted/unweighted internal va...,passed
2,ESS4/ESS8 revised missingness handling and CCA...,passed
3,ESS8 historical CCA subset against Supplementa...,passed
4,ESS4 historical respondent subset against df_E...,passed
5,ESS4 historical demographics against df_ESS4.R...,passed
6,ESS4 historical beliefs against df_ESS4.RData,passed
7,ESS4/ESS8 full-sample shared-belief descriptiv...,passed


Validation completed successfully.

Principal sample sizes:
- ESS4: 55,044 respondents
- ESS8: 43,148 respondents

Historical validation subset sizes:
- ESS4: 45,268 respondents
- ESS8: 37,118 respondents

Generated validation files:
- /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess4_cca_subset_validation_against_reference.csv
- /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess8_cca_subset_validation_against_paper.csv
- /Users/karan/Desktop/SSM-MERC/polarization/Github/C72H-SimPol/data_curation/data/processed/ess4_ess8_shared_belief_descriptives.csv
